Imports

In [1]:
import os
import json
import re
import time
import pandas as pd
import spacy
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp
import joblib
import torch
from torchvision.models import resnet101, ResNet101_Weights
from torch.utils.data import Dataset, DataLoader
from PIL import Image

1-Data Preparation

In [4]:
mo_df = pd.read_json(r"..\data\modcloth_final_data.json", lines=True)
re_df = pd.read_json(r"..\data\renttherunway_final_data.json", lines=True)
st_df = pd.read_csv(r"..\data\styles.csv", on_bad_lines="skip")

In [5]:
st_df["baseColour"] = st_df["baseColour"].fillna(st_df["baseColour"].mode()[0])
st_df["season"] = st_df["season"].fillna(st_df["season"].mode()[0])
st_df["year"] = st_df["year"].fillna(st_df["year"].median())
st_df["usage"] = st_df["usage"].fillna(st_df["usage"].mode()[0])
st_df["productDisplayName"] = st_df["productDisplayName"].fillna("Unknown")

In [6]:
re_df["age"] = re_df["age"].fillna(re_df["age"].median())
re_df["height"] = re_df["height"].fillna(re_df["height"].mode()[0])
re_df["body type"] = re_df["body type"].fillna(re_df["body type"].mode()[0])
re_df["review_summary"] = re_df["review_summary"].fillna("No Summary")
re_df["weight"] = re_df["weight"].fillna(re_df["weight"].mode()[0])
re_df["bust size"] = re_df["bust size"].fillna(re_df["bust size"].mode()[0])
re_df["rating"] = re_df["rating"].fillna(re_df["rating"].median())
re_df["rented for"] = re_df["rented for"].fillna(re_df["rented for"].mode()[0])
re_df["review_text"] = re_df["review_text"].fillna("No Review")

In [7]:
def convert_height(x):
    if pd.isna(x):
        return None
    feet = int(x.split("'")[0])
    inches = int(x.split("'")[1].replace('"', '').strip())
    cm = (feet * 30.48) + (inches * 2.54)
    return round(cm)

re_df["height_cm"] = re_df["height"].apply(convert_height)
re_df.drop("height", axis=1, inplace=True)

In [8]:
re_df["weight_lbs"] = re_df["weight"].str.replace("lbs", "").astype(int)
re_df["weight_kg"] = (
    re_df["weight"]
    .str.replace("lbs", "")
    .astype(int) * 0.453592
)
re_df["weight_kg"] = re_df["weight_kg"].round().astype(int)
re_df.drop("weight", axis=1, inplace=True)
re_df.drop("weight_lbs", axis=1, inplace=True)

In [9]:
re_df = re_df.drop_duplicates()

In [10]:
re_df.loc[re_df["height_cm"] > 200, "height_cm"] = None
re_df["height_cm"] = re_df["height_cm"].fillna(re_df["height_cm"].median())

In [11]:
re_df.loc[re_df["age"] > 80, "age"] = None
re_df["age"] = re_df["age"].fillna(re_df["age"].median())

In [12]:
def height_to_cm(x):
    if pd.isna(x):
        return None
    x = x.replace("ft", "").replace("in", "")
    parts = x.split()
    feet = int(parts[0])
    inches = int(parts[1]) if len(parts) > 1 else 0
    return round((feet * 30.48) + (inches * 2.54))

mo_df["height_cm"] = mo_df["height"].apply(height_to_cm)
mo_df.drop("height", axis=1, inplace=True)

In [13]:
mo_df["review_summary"] = mo_df["review_summary"].fillna("No Summary")
mo_df["review_text"] = mo_df["review_text"].fillna("No Review")
mo_df["height_cm"] = mo_df["height_cm"].fillna(mo_df["height_cm"].median())
mo_df["cup size"] = mo_df["cup size"].fillna(mo_df["cup size"].mode()[0])
mo_df["length"] = mo_df["length"].fillna(mo_df["length"].mode()[0])
mo_df["quality"] = mo_df["quality"].fillna(mo_df["quality"].mode()[0])

In [14]:
mo_df = mo_df.drop_duplicates()

In [15]:
mo_df = mo_df[(mo_df["height_cm"] >= 140) & (mo_df["height_cm"] <= 210)]
mo_df.loc[mo_df["size"] == 0, "size"] = mo_df["size"].median()

median_height = mo_df["height_cm"].median()
mo_df.loc[
    (mo_df["height_cm"] < 140) |
    (mo_df["height_cm"] > 210),
    "height_cm"
] = median_height

mo_df.loc[
    (mo_df["height_cm"] < 140) |
    (mo_df["height_cm"] > 210),
    "height_cm"
] = mo_df["height_cm"].median()

In [16]:
re_df["source"] = "renttherunway"
mo_df["source"] = "modcloth"

print("re_df columns:", list(re_df.columns))
print("mo_df columns:", list(mo_df.columns))

merged_df = pd.concat([re_df, mo_df], ignore_index=True, sort=False)

print("re_df shape:", re_df.shape)
print("mo_df shape:", mo_df.shape)
print("merged_df shape:", merged_df.shape)

re_df columns: ['fit', 'user_id', 'bust size', 'item_id', 'rating', 'rented for', 'review_text', 'body type', 'review_summary', 'category', 'size', 'age', 'review_date', 'height_cm', 'weight_kg', 'source']
mo_df columns: ['item_id', 'waist', 'size', 'quality', 'cup size', 'hips', 'bra size', 'category', 'bust', 'user_name', 'length', 'fit', 'user_id', 'shoe size', 'shoe width', 'review_summary', 'review_text', 'height_cm', 'source']
re_df shape: (192355, 16)
mo_df shape: (82349, 19)
merged_df shape: (274704, 26)


2-Visual Feature Extraction & Fusion

In [18]:
image_dir = r"..\data\images"
jpg_files = [
    f for f in os.listdir(image_dir)
    if f.lower().endswith(".jpg")
]

image_df = pd.DataFrame({"filename": jpg_files})

image_df["id"] = pd.to_numeric(
    image_df["filename"].str.replace(".jpg", "", regex=False),
    errors="coerce"
)

image_df = image_df.merge(
    st_df,
    on="id",
    how="inner"
)

print("Images linked to styles:", len(image_df))

Images linked to styles: 44419


In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights = ResNet101_Weights.DEFAULT

model = resnet101(weights=weights)
model = model.to(device)
model.eval()

transform = weights.transforms()

print("Using device:", device)

Using device: cuda


In [20]:
feature_extractor = torch.nn.Sequential(
    *list(model.children())[:-1]
)

feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, ker

In [21]:
class ClothingImageDataset(Dataset):

    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image_path = os.path.join(
            self.image_dir,
            row["filename"]
        )

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, row["id"], row["articleType"]

In [22]:
dataset = ClothingImageDataset(
    image_df,
    image_dir,
    transform=transform
)

dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

print("Number of images:", len(dataset))

Number of images: 44419


In [23]:
all_features = []
all_ids = []
all_categories = []

with torch.no_grad():

    for images, ids, categories in dataloader:

        images = images.to(device)

        features = feature_extractor(images)

        features = features.flatten(1)

        all_features.append(features.cpu())
        all_ids.extend(ids.numpy())
        all_categories.extend(categories)

print("Feature extraction complete.")

Feature extraction complete.


In [24]:
features_tensor = torch.cat(all_features, dim=0)

features_df = pd.DataFrame(
    features_tensor.numpy(),
    columns=[f"image_feature_{i+1}" for i in range(2048)]
)

features_df["id"] = all_ids
features_df["articleType"] = all_categories

print("Feature shape:", features_df.shape)

Feature shape: (44419, 2050)


In [25]:
image_feature_columns = [
    f"image_feature_{i+1}" for i in range(2048)
]

category_features = features_df.groupby(
    "articleType"
)[image_feature_columns].mean()

print("Category feature shape:", category_features.shape)

Category feature shape: (142, 2048)


In [26]:
category_mapping = {
    "dress": "Dresses", "dresses": "Dresses", "gown": "Dresses", "ballgown": "Dresses",
    "sheath": "Dresses", "shift": "Dresses", "shirtdress": "Dresses", "frock": "Dresses",
    "maxi": "Dresses", "midi": "Dresses", "mini": "Dresses", "wedding": "Dresses",

    "top": "Tops", "tops": "Tops", "blouse": "Tops", "tank": "Tops", "turtleneck": "Tops",

    "tee": "Tshirts", "t-shirt": "Tshirts", "henley": "Tshirts", "crewneck": "Tshirts",

    "shirt": "Shirts", "buttondown": "Shirts",

    "sweater": "Sweaters", "cardigan": "Sweaters", "knit": "Sweaters", "pullover": "Sweaters",

    "sweatshirt": "Sweatshirts", "sweatershirt": "Sweatshirts", "hoodie": "Sweatshirts",

    "pants": "Trousers", "pant": "Trousers", "trouser": "Trousers", "trousers": "Trousers",
    "culotte": "Trousers", "culottes": "Trousers", "bottoms": "Trousers",

    "jeans": "Jeans",

    "legging": "Leggings", "leggings": "Leggings",

    "jacket": "Jackets", "outerwear": "Jackets", "bomber": "Jackets", "parka": "Jackets",
    "trench": "Jackets", "duster": "Jackets", "blouson": "Jackets", "coat": "Jackets",
    "peacoat": "Jackets", "overcoat": "Jackets",

    "blazer": "Blazers",

    "skirt": "Skirts", "skirts": "Skirts", "skort": "Skirts",

    "jumpsuit": "Jumpsuit", "romper": "Rompers", "combo": "Clothing Set", "overalls": "Clothing Set",

    "tunic": "Tunics", "kimono": "Tunics", "poncho": "Tunics", "cape": "Tunics",

    "kaftan": "Kurtas", "caftan": "Kurtas",

    "vest": "Waistcoat",

    "suit": "Clothing Set", "cami": "Camisoles", "tight": "Tights",
    "jogger": "Track Pants", "sweatpants": "Track Pants",
}

In [27]:
merged_df["image_category"] = merged_df["category"].map(category_mapping)

print("Total reviews:", len(merged_df))
print("Mapped:", merged_df["image_category"].notna().sum())
print("Unmapped:", merged_df["image_category"].isna().sum())

category_features.to_csv("../data/category_image_features.csv")

merged_df.to_parquet("../data/merged_with_image_category.parquet", index=False)

print("Phase 2 files saved successfully.")
print("category_features shape:", category_features.shape)
print("merged_df shape:", merged_df.shape)

Total reviews: 274704
Mapped: 250229
Unmapped: 24475
Phase 2 files saved successfully.
category_features shape: (142, 2048)
merged_df shape: (274704, 27)


3-Text Feature Extraction

In [29]:
final_df = pd.read_parquet("../data/merged_with_image_category.parquet")
print(final_df.shape)

(274704, 27)


In [31]:
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def clean_text(text):
    if not isinstance(text, str):
        return ""
    return re.sub(r"[^a-zA-Z\s]", "", text.lower())

pre_cleaned = [clean_text(t) for t in final_df["review_text"]]

cleaned = []
t0 = time.time()
for doc in tqdm(nlp.pipe(pre_cleaned, batch_size=500), total=len(pre_cleaned), desc="Cleaning reviews"):
    tokens = [token.lemma_ for token in doc if not token.is_stop and token.lemma_.strip()]
    cleaned.append(" ".join(tokens))

final_df["clean_review"] = cleaned
print(f"Done | {time.time()-t0:.1f}s")

Cleaning reviews: 100%|██████████| 274704/274704 [12:47<00:00, 357.81it/s] 

Done | 767.8s


In [32]:
MAX_TFIDF_FEATURES = 300

vectorizer = TfidfVectorizer(max_features=MAX_TFIDF_FEATURES)
tfidf_matrix = vectorizer.fit_transform(final_df["clean_review"])
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

TF-IDF matrix shape: (274704, 300)


In [33]:
sp.save_npz("../data/text_features.npz", tfidf_matrix)
joblib.dump(vectorizer, "../data/tfidf_vectorizer.pkl")

final_df.to_parquet("../data/final_df_with_clean_text.parquet")

print("Saved: text_features.npz, tfidf_vectorizer.pkl, final_df_with_clean_text.parquet")


Saved: text_features.npz, tfidf_vectorizer.pkl, final_df_with_clean_text.parquet


4-Feature Fusion

5-Model Training

6-Deployment

7-Model Evaluation